# Speed Up Python Code and LLM Applications with Caching

Companion notebook for the To Data & Beyond tutorial [How to Use Caching to Speed Up Your Python Code & LLM Application](https://todatabeyond.com/blog/how-to-use-caching-to-speed-up-your-python-code-and-llm-application).

The standard-library examples require Python 3.9 or later and run without credentials. The final GPTCache section preserves the article's July 2024 legacy integration as an optional archival example.

## 1. Recursive Fibonacci without caching

This deliberately recursive implementation repeats the same work many times.

In [ ]:
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)


fibonacci(10)

## 2. Unbounded caching with `functools.cache`

`@cache` memoizes each result by arguments and reuses it on later calls.

In [ ]:
from functools import cache


@cache
def fibonacci_cached(n):
    if n <= 1:
        return n
    return fibonacci_cached(n - 1) + fibonacci_cached(n - 2)


fibonacci_cached(10)

## 3. Bounded caching with `functools.lru_cache`

A bounded LRU cache limits memory use by evicting the least recently used entries.

In [ ]:
from functools import lru_cache


@lru_cache(maxsize=7)
def fibonacci_limited(n):
    if n <= 1:
        return n
    return fibonacci_limited(n - 1) + fibonacci_limited(n - 2)


fibonacci_limited(5)
fibonacci_limited(3)
fibonacci_limited.cache_info()

## 4. Compare execution time

Timings depend on the runtime. Clear both caches before measuring so rerunning the cell gives a fair first-call comparison.

In [ ]:
from functools import cache, lru_cache
import timeit


def fibonacci_no_cache(n):
    if n <= 1:
        return n
    return fibonacci_no_cache(n - 1) + fibonacci_no_cache(n - 2)


@cache
def fibonacci_cache(n):
    if n <= 1:
        return n
    return fibonacci_cache(n - 1) + fibonacci_cache(n - 2)


@lru_cache
def fibonacci_lru_cache(n):
    if n <= 1:
        return n
    return fibonacci_lru_cache(n - 1) + fibonacci_lru_cache(n - 2)

In [ ]:
n = 35
fibonacci_cache.cache_clear()
fibonacci_lru_cache.cache_clear()

no_cache_time = timeit.timeit(lambda: fibonacci_no_cache(n), number=1)
cache_time = timeit.timeit(lambda: fibonacci_cache(n), number=1)
lru_cache_time = timeit.timeit(lambda: fibonacci_lru_cache(n), number=1)

print(f"Time without cache: {no_cache_time:.6f} seconds")
print(f"Time with cache: {cache_time:.6f} seconds")
print(f"Time with LRU cache: {lru_cache_time:.6f} seconds")

## 5. Optional archival GPTCache example

> Compatibility note: this is the original July 2024 integration. It pins `openai==0.28.1`, uses the retired `ChatCompletion` interface, and may conflict with packages in a modern notebook runtime. Run it only in a fresh disposable runtime after reviewing the current [GPTCache repository](https://github.com/zilliztech/GPTCache). It requires an OpenAI API key and incurs API usage.

Store the key in Colab Secrets as `OPENAI_API_KEY`, or set it in your environment before starting the notebook. Never paste or save the key in a cell.

In [ ]:
# Optional: run only in a fresh runtime for the archived example.
# %pip install -q gptcache "openai==0.28.1"

In [ ]:
import os
import time


def response_text(openai_resp):
    return openai_resp["choices"][0]["message"]["content"]


if "OPENAI_API_KEY" not in os.environ:
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except (ImportError, TypeError):
        raise RuntimeError(
            "Set OPENAI_API_KEY in the environment or Colab Secrets before running this cell."
        )

from gptcache import cache
from gptcache.adapter import openai

cache.init()
cache.set_openai_key()

question = "what's github"
for _ in range(2):
    start_time = time.time()
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": question}],
    )
    print(f"Question: {question}")
    print("Time consuming: {:.2f}s".format(time.time() - start_time))
    print(f"Answer: {response_text(response)}\n")